<a href="https://colab.research.google.com/github/CelBabe6656/SoloStock2/blob/main/SoloStock.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas

import pandas as pd
from dataclasses import dataclass, asdict
from typing import List, Dict, Optional
import uuid
from datetime import datetime


In [2]:
gl_accounts_data = [
    ("INC-01", "Sole Trader Sales", "Revenue", "Revenue", "P8"),
    ("INC-02", "PAYG Gross Wages", "Revenue", "Revenue", ""),
    ("EXP-01", "Materials & Supplies", "COGS", "COGS", "P8 (C)"),
    ("EXP-02", "Subcontractor Fees", "COGS", "COGS", "P8 (C)"),
    ("EXP-03", "Motor Vehicle - Fuel", "Expense", "Operating Expenses", "D1/P8"),
    ("EXP-04", "Motor Vehicle - Other", "Expense", "Operating Expenses", "D1/P8"),
    ("EXP-05", "Small Tools (<$1,000)", "Expense", "Operating Expenses", "D2"),
    ("EXP-06", "Uniforms & Protective", "Expense", "Operating Expenses", "D3"),
    ("AST-01", "Plant & Equipment", "Asset", "Capital Assets", "P8"),
    ("AST-02", "Motor Vehicle Asset", "Asset", "Capital Assets", "P8"),
    ("AST-POOL", "Simplified Depreciation Pool", "Asset", "Capital Assets", "P8"),
    ("LIA-01", "GST Collected", "Liability", "Tax & Liabilities", ""),
    ("LIA-02", "GST Paid", "Liability", "Tax & Liabilities", ""),
    ("LIA-03", "Income Tax Reserve", "Liability", "Tax & Liabilities", ""),
    ("EQ-01", "Owner's Capital", "Equity", "Equity", ""),
    ("EQ-02", "Drawings", "Equity", "Equity", ""),
]

gl_accounts = pd.DataFrame(
    gl_accounts_data,
    columns=["gl_code", "name", "type", "category", "ato_label"]
)

gl_accounts


,gl_code,name,type,category,ato_label
0,INC-01,Sole Trader Sales,Revenue,Revenue,P8
1,INC-02,PAYG Gross Wages,Revenue,Revenue,
2,EXP-01,Materials & Supplies,COGS,COGS,P8 (C)
3,EXP-02,Subcontractor Fees,COGS,COGS,P8 (C)
4,EXP-03,Motor Vehicle - Fuel,Expense,Operating Expenses,D1/P8
5,EXP-04,Motor Vehicle - Other,Expense,Operating Expenses,D1/P8
6,EXP-05,"Small Tools (<$1,000)",Expense,Operating Expenses,D2
7,EXP-06,Uniforms & Protective,Expense,Operating Expenses,D3
8,AST-01,Plant & Equipment,Asset,Capital Assets,P8
9,AST-02,Motor Vehicle Asset,Asset,Capital Assets,P8


In [3]:
def calculate_income_tax_au(taxable_income: float) -> float:
    if taxable_income <= 0:
        return 0.0

    brackets = [
        (0,       18200,  0.0,    0.0),
        (18200,   45000,  0.0,    0.19),
        (45000,   135000, 5092.0, 0.325),
        (135000,  190000, 29467.0,0.37),
        (190000,  float("inf"), 51667.0, 0.45),
    ]

    for lower, upper, base_tax, rate in brackets:
        if lower < taxable_income <= upper:
            return base_tax + (taxable_income - lower) * rate

    return 0.0


def calculate_tax_summary(projected_payg_gross: float,
                          ytd_sole_profit: float,
                          payg_withheld: float,
                          weeks_remaining: int) -> Dict:
    total_taxable_income = projected_payg_gross + ytd_sole_profit
    income_tax = calculate_income_tax_au(total_taxable_income)
    medicare_levy = 0.02 * total_taxable_income
    estimated_total_tax = income_tax + medicare_levy
    tax_shortfall = max(0.0, estimated_total_tax - payg_withheld)
    weekly_saving_rate = tax_shortfall / weeks_remaining if weeks_remaining > 0 else tax_shortfall

    return {
        "total_taxable_income": total_taxable_income,
        "income_tax": income_tax,
        "medicare_levy": medicare_levy,
        "estimated_total_tax": estimated_total_tax,
        "tax_shortfall": tax_shortfall,
        "weekly_saving_rate": weekly_saving_rate,
    }

# quick demo
calculate_tax_summary(80000, 20000, 15000, 20)


{'total_taxable_income': 100000,
 'income_tax': 22967.0,
 'medicare_levy': 2000.0,
 'estimated_total_tax': 24967.0,
 'tax_shortfall': 9967.0,
 'weekly_saving_rate': 498.35}

In [4]:
@dataclass
class Transaction:
    id: str
    date: str
    vendor: str
    description: str
    total_amount: float
    gst_amount: float
    source_type: str  # "PAYG", "SOLE_TRADER", "BOTH"
    business_percent: float
    payg_percent: float
    personal_percent: float
    gl_code: str
    ato_label: str
    is_asset: bool
    receipt_id: Optional[str]
    confidence: float
    needs_review: bool


transactions: List[Transaction] = []  # in-memory list


def add_transaction(**kwargs) -> Transaction:
    tx = Transaction(
        id=str(uuid.uuid4()),
        **kwargs
    )
    transactions.append(tx)
    return tx


def transactions_to_df() -> pd.DataFrame:
    return pd.DataFrame([asdict(t) for t in transactions])


In [5]:
def simple_gl_mapper(description: str, is_asset: bool = False) -> str:
    desc = description.lower()
    if "fuel" in desc or "petrol" in desc:
        return "EXP-03"
    if "rego" in desc or "registration" in desc or "insurance" in desc:
        return "EXP-04"
    if "bunnings" in desc or "tool" in desc:
        # if asset, send to plant & equipment
        return "AST-01" if is_asset else "EXP-05"
    if "uniform" in desc or "hi vis" in desc:
        return "EXP-06"
    # default catch-all
    return "EXP-01"


In [6]:
# Example: Sole trader fuel
add_transaction(
    date="2025-07-10",
    vendor="Shell",
    description="Fuel for ute",
    total_amount=120.0,
    gst_amount=10.91,
    source_type="SOLE_TRADER",
    business_percent=80.0,
    payg_percent=0.0,
    personal_percent=20.0,
    gl_code=simple_gl_mapper("Fuel for ute"),
    ato_label="P8",
    is_asset=False,
    receipt_id="RCP-001",
    confidence=0.95,
    needs_review=False,
)

# Example: Ladder $1,200 used for both
add_transaction(
    date="2025-07-12",
    vendor="Bunnings",
    description="Ladder 4m",
    total_amount=1200.0,
    gst_amount=109.09,
    source_type="BOTH",
    business_percent=60.0,
    payg_percent=40.0,
    personal_percent=0.0,
    gl_code=simple_gl_mapper("Bunnings ladder", is_asset=True),
    ato_label="P8",
    is_asset=True,
    receipt_id="RCP-002",
    confidence=0.92,
    needs_review=False,
)

transactions_to_df()


,id,date,vendor,description,total_amount,gst_amount,source_type,business_percent,payg_percent,personal_percent,gl_code,ato_label,is_asset,receipt_id,confidence,needs_review
0,94c96508-b34e-4e31-adba-a7c2f09d28e4,2025-07-10,Shell,Fuel for ute,120.0,10.91,SOLE_TRADER,80.0,0.0,20.0,EXP-03,P8,False,RCP-001,0.95,False
1,7ef76f57-df59-4f76-b74f-2d78d82e046f,2025-07-12,Bunnings,Ladder 4m,1200.0,109.09,BOTH,60.0,40.0,0.0,AST-01,P8,True,RCP-002,0.92,False


In [7]:
current_cash_at_bank = 15000.0  # pretend user entered this


def can_afford_purchase(purchase_amount: float,
                        projected_payg_gross: float,
                        ytd_sole_profit: float,
                        payg_withheld: float,
                        weeks_remaining: int) -> Dict:
    tax_summary = calculate_tax_summary(
        projected_payg_gross,
        ytd_sole_profit,
        payg_withheld,
        weeks_remaining
    )
    projected_tax_bill = tax_summary["estimated_total_tax"]
    # very simple: reserve tax in LIA-03, rest is "safe"
    safe_to_spend = max(0.0, current_cash_at_bank - projected_tax_bill)

    return {
        "projected_tax_bill": projected_tax_bill,
        "safe_to_spend": safe_to_spend,
        "can_afford": purchase_amount <= safe_to_spend
    }


can_afford_purchase(
    purchase_amount=2000.0,
    projected_payg_gross=80000,
    ytd_sole_profit=20000,
    payg_withheld=15000,
    weeks_remaining=20
)


{'projected_tax_bill': 24967.0, 'safe_to_spend': 0.0, 'can_afford': False}

In [8]:
vehicle_trips: List[Dict] = []


def add_vehicle_trip(date: str, km: float, purpose_source: str, logbook_percent: float = 0.0, notes: str = ""):
    trip = {
        "id": str(uuid.uuid4()),
        "date": date,
        "km": km,
        "purpose_source": purpose_source,  # "PAYG" or "SOLE_TRADER"
        "logbook_percent": logbook_percent,
        "notes": notes,
    }
    vehicle_trips.append(trip)
    return trip


def vehicle_km_summary():
    df = pd.DataFrame(vehicle_trips)
    if df.empty:
        return {"payg_km": 0.0, "business_km": 0.0}
    payg_km = df[df["purpose_source"] == "PAYG"]["km"].sum()
    business_km = df[df["purpose_source"] == "SOLE_TRADER"]["km"].sum()
    return {"payg_km": payg_km, "business_km": business_km}


# demo
add_vehicle_trip("2025-07-01", 50, "PAYG")
add_vehicle_trip("2025-07-02", 80, "SOLE_TRADER", logbook_percent=70.0)
vehicle_km_summary()


{'payg_km': np.int64(50), 'business_km': np.int64(80)}

In [9]:
def check_payg_km_cap():
    summary = vehicle_km_summary()
    if summary["payg_km"] >= 4500:
        return "Warning: You are approaching the 5,000km PAYG cap for cents-per-km."
    return "PAYG km within safe range."

check_payg_km_cap()


'PAYG km within safe range.'

In [10]:
def export_gl_report_csv(filename: str = "gl_report.csv"):
    df = transactions_to_df()
    df.to_csv(filename, index=False)
    return filename

export_gl_report_csv()


'gl_report.csv'

In [11]:
def get_total_spent_by_vendor(vendor_name: str) -> float:
    df = transactions_to_df()
    if df.empty:
        return 0.0
    mask = df["vendor"].str.lower() == vendor_name.lower()
    return df[mask]["total_amount"].sum()


def chat_agent(user_message: str) -> str:
    msg = user_message.lower()

    if "bunnings" in msg and "spend" in msg:
        total = get_total_spent_by_vendor("Bunnings")
        return f"You have spent ${total:.2f} at Bunnings in the data currently loaded."

    if "fuel" in msg and "quarter" in msg:
        df = transactions_to_df()
        if df.empty:
            return "No transactions loaded."
        fuel = df[df["gl_code"] == "EXP-03"]["total_amount"].sum()
        return f"You have spent ${fuel:.2f} on fuel (Motor Vehicle - Fuel)."

    return "In a full system, I would call more tools to answer this. For now, I only handle Bunnings and fuel questions."


print(chat_agent("How much did I spend at Bunnings?"))
print(chat_agent("How much have I spent on fuel this quarter?"))


You have spent $1200.00 at Bunnings in the data currently loaded.
You have spent $120.00 on fuel (Motor Vehicle - Fuel).
